# 05.1 – Modeling Attrition Prediction with XGBoost

## 🎯 Mục tiêu

Notebook này thực hiện **xây dựng mô hình dự đoán Attrition** bằng thuật toán **XGBoost**, với các mục tiêu cụ thể:

- Xây dựng mô hình **XGBoost** cho bài toán phân lớp nhị phân (**Leave / Stay**)
- Chỉ sử dụng dữ liệu đã tiền xử lý: `hr_processed_ml.csv`
- Ghi nhận đầy đủ:
  - Thiết lập thực nghiệm
  - Hyperparameters
  - Thời gian huấn luyện
  - Các metric đánh giá phù hợp
- Lưu **model, scaler và kết quả dự đoán** để phục vụ so sánh ở notebook **Evaluation**

In [ ]:
# ================================
# Import các thư viện hệ thống
# ================================
import os
import time
import json
import joblib

# ================================
# Xử lý dữ liệu
# ================================
import numpy as np
import pandas as pd

# ================================
# Trực quan hóa
# ================================
import matplotlib.pyplot as plt
import seaborn as sns

# ================================
# Scikit-learn
# ================================
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    recall_score,
    precision_score,
    average_precision_score
)

# ================================
# XGBoost
# ================================
from xgboost import XGBClassifier

In [ ]:
# ================================
# Cấu hình chung
# ================================
sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

RANDOM_STATE = 42
# (cải thiện phát hiện Leave)
THRESHOLD_RANGE = np.arange(0.1, 0.6, 0.02)   # ⚠️ Giảm threshold để tăng Recall Leave
TOP_K_RATIO = 0.2                            

In [ ]:
# ================================
# Đọc dữ liệu đã preprocess
# ================================
DATA_PATH = "../data/processed/hr_processed_ml.csv"

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
df.head()

# ================================
# Cấu hình thư mục lưu kết quả
# ================================

BASE_MODEL_DIR = "../data/processed/models/xgb"

# Tạo folder nếu chưa tồn tại
os.makedirs(BASE_MODEL_DIR, exist_ok=True)

# Định nghĩa các đường dẫn dùng chung
MODEL_PATH   = os.path.join(BASE_MODEL_DIR, "xgb_model.pkl")
SCALER_PATH  = os.path.join(BASE_MODEL_DIR, "xgb_scaler.pkl")
METRICS_PATH = os.path.join(BASE_MODEL_DIR, "xgb_metrics.json")
PRED_PATH    = os.path.join(BASE_MODEL_DIR, "xgb_predictions.csv")

print("Artifacts will be saved to:", BASE_MODEL_DIR)

In [ ]:
# ================================
# Xác định biến mục tiêu
# ================================
TARGET_COL = "Attrition"

# ================================
# Tách X và y
# ================================
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

print("X shape:", X.shape)
print("Phân bố nhãn y:")
print(y.value_counts(normalize=True))

In [ ]:
# ================================
# Chia dữ liệu train / test
# ================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Train size:", X_train.shape)
print("Test size :", X_test.shape)

In [ ]:
# ================================
# Xử lý mất cân bằng lớp (Leave / Stay)
# ================================
neg, pos = np.bincount(y_train)
scale_pos_weight = neg / pos

print("scale_pos_weight:", round(scale_pos_weight, 2))

In [ ]:
# ================================
# Chuẩn hoá dữ liệu
# ================================
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

In [ ]:
# ================================
# Khởi tạo mô hình XGBoost
# ================================
xgb_model = XGBClassifier(
    n_estimators=300,          # số cây boosting
    max_depth=5,               # độ sâu cây
    learning_rate=0.05,        # tốc độ học
    subsample=0.8,             # sampling theo hàng
    colsample_bytree=0.8,      # sampling theo cột
    scale_pos_weight=scale_pos_weight,
    objective="binary:logistic",
    eval_metric="aucpr",
    random_state=RANDOM_STATE,
    n_jobs=-1                  # dùng toàn bộ CPU
)

In [ ]:
# ================================
# Huấn luyện mô hình
# ================================
start_time = time.time()

xgb_model.fit(
    X_train_scaled,
    y_train,
    eval_set=[(X_test_scaled, y_test)],
    verbose=False
)

train_time = time.time() - start_time
print(f"Training time: {train_time:.2f} seconds")

In [ ]:
# ================================
# Dự đoán xác suất lớp Leave (cải thiện phát hiện Leave)
# ================================
y_proba = xgb_model.predict_proba(X_test_scaled)[:, 1]

# ================================
# Tối ưu threshold theo Recall Leave
# ================================
best_threshold = 0.5
best_recall = 0

for t in THRESHOLD_RANGE:
    y_tmp = (y_proba >= t).astype(int)
    recall_tmp = recall_score(y_test, y_tmp)

    if recall_tmp > best_recall:
        best_recall = recall_tmp
        best_threshold = t

print("Best threshold:", best_threshold)
print("Best Recall:", round(best_recall, 4))

In [ ]:
# ================================
# Áp dụng threshold tối ưu để dự đoán nhãn (cải thiện phát hiện Leave)
# ================================
y_pred = (y_proba >= best_threshold).astype(int)

In [ ]:
# ================================
# Đánh giá (Leave-focused)
# ================================
pr_auc = average_precision_score(y_test, y_proba)
f1 = f1_score(y_test, y_pred)
recall_leave = recall_score(y_test, y_pred)
precision_leave = precision_score(y_test, y_pred)

print("PR-AUC :", round(pr_auc, 4))
print("F1-score:", round(f1, 4))
print("Recall (Leave):", round(recall_leave, 4))
print("Precision (Leave):", round(precision_leave, 4))

In [ ]:
# ================================
# Recall@TopK (cải thiện phát hiện Leave)
# ================================
top_k = int(len(y_test) * TOP_K_RATIO)
top_k_idx = np.argsort(y_proba)[-top_k:]
recall_at_k = y_test.iloc[top_k_idx].sum() / y_test.sum()

print(f"Recall@Top {int(TOP_K_RATIO*100)}%:", round(recall_at_k, 4))

In [ ]:
# ================================
# Báo cáo phân lớp chi tiết
# ================================
print(
    classification_report(
        y_test,
        y_pred,
        target_names=["Stay", "Leave"]
    )
)

In [ ]:
# ================================
# Vẽ Confusion Matrix
# ================================
cm = confusion_matrix(y_test, y_pred)

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Stay", "Leave"],
    yticklabels=["Stay", "Leave"]
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix – XGBoost")
plt.tight_layout()
plt.show()

In [ ]:
# ================================
# Lưu kết quả dự đoán
# ================================
# Tạo DataFrame chứa kết quả dự đoán
pred_df = pd.DataFrame({
    "y_true": y_test.values,   # Nhãn thực tế
    "y_proba": y_proba,        # Xác suất Leave
    "y_pred": y_pred           # Nhãn dự đoán
})

# Lưu ra file CSV
pred_df.to_csv(PRED_PATH, index=False)

print("Saved predictions to:", PRED_PATH)

In [ ]:
# ================================
# Lưu các metric và hyperparameters
# ================================
metrics = {
    # Performance metrics
    "pr_auc": float(pr_auc),
    "recall_leave": float(recall_leave),
    "precision_leave": float(precision_leave),
    "f1_leave": float(f1),
    "threshold_used": float(best_threshold),
    "train_time_sec": float(train_time),

    # Hyperparameters
    "n_estimators": 300,
    "max_depth": 5,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": RANDOM_STATE
}

# Lưu metrics ra file JSON
with open(METRICS_PATH, "w") as f:
    json.dump(metrics, f, indent=4)

print("Saved metrics to:", METRICS_PATH)

In [ ]:
# ================================
# Lưu model và scaler
# ================================
# Lưu XGBoost model
joblib.dump(xgb_model, MODEL_PATH)

# Lưu scaler
joblib.dump(scaler, SCALER_PATH)

print("Saved model  to:", MODEL_PATH)
print("Saved scaler to:", SCALER_PATH)